# Notebook 01 — Building the feature matrix

## Context
The project works on the *Home Credit Default Risk* dataset, which arrives as several tables: the loan application itself, the credit bureau records, previous applications, and so on.

The approach is the standard one for this dataset:
- **one table, one preprocessing and aggregation function**
- **progressive joins** on the `SK_ID_CURR` key
- one **tabular feature matrix** at the end, ready for training

## What this notebook does
- Runs the reproducible feature-engineering pipeline that lives in `src/credexp/data/build_features.py`
- Writes `data/processed/features.parquet`
- Checks the resulting shape, and the train/test split carried by `TARGET`

> The heavy lifting sits in a Python module rather than in the notebook. A notebook keeps state between cells and loses it between sessions, so anything that has to be reproducible does not belong in one.

In [1]:
from pathlib import Path

import pandas as pd

from credexp.config import DATA_DIR
from credexp.data.build_features import build_feature_matrix
from credexp.data.io import load_parquet, save_parquet


## Prerequisite: the raw tables

The Kaggle files must be under `data/raw/`, under exactly these names:

- application_train.csv
- application_test.csv
- bureau.csv
- bureau_balance.csv
- previous_application.csv
- POS_CASH_balance.csv
- installments_payments.csv
- credit_card_balance.csv

In [2]:
raw_dir = DATA_DIR / "raw"
required = [
    "application_train.csv",
    "application_test.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "POS_CASH_balance.csv",
    "installments_payments.csv",
    "credit_card_balance.csv",
]

missing = [f for f in required if not (raw_dir / f).exists()]
missing


[]

An empty `missing` list means the feature matrix can be built.

A debug mode running on a subset is available for a quick check. What follows is the full pass, the one that produces the matrix training actually uses.

In [3]:
out_path = DATA_DIR / "processed" / "features.parquet"

df = build_feature_matrix(raw_path=raw_dir, debug=False)
save_parquet(df, out_path)

df.shape


{"ts": "2026-02-18T19:51:23Z", "level": "INFO", "name": "credexp.data.io", "msg": "read_csv path=data\\raw\\application_train.csv"}
{"ts": "2026-02-18T19:51:25Z", "level": "INFO", "name": "credexp.data.io", "msg": "read_csv path=data\\raw\\application_test.csv"}
{"ts": "2026-02-18T19:51:25Z", "level": "INFO", "name": "credexp.data.build_features", "msg": "Train samples: 307511, test samples: 48744"}
{"ts": "2026-02-18T19:51:26Z", "level": "INFO", "name": "credexp.data.io", "msg": "read_csv path=data\\raw\\bureau.csv"}
{"ts": "2026-02-18T19:51:28Z", "level": "INFO", "name": "credexp.data.io", "msg": "read_csv path=data\\raw\\bureau_balance.csv"}
{"ts": "2026-02-18T19:51:38Z", "level": "INFO", "name": "credexp.data.build_features", "msg": "Bureau df shape: (305811, 116)"}
{"ts": "2026-02-18T19:51:38Z", "level": "INFO", "name": "credexp.data.build_features", "msg": "Process bureau and bureau_balance - done in 12s"}
{"ts": "2026-02-18T19:51:38Z", "level": "INFO", "name": "credexp.data.io",

(356251, 797)

In [4]:
df2 = load_parquet(out_path)
df2.shape, df2.columns[:10]


{"ts": "2026-02-18T19:52:38Z", "level": "INFO", "name": "credexp.data.io", "msg": "load_parquet path=data\\processed\\features.parquet"}


((356251, 797),
 Index(['SK_ID_CURR', 'TARGET', 'CODE_GENDER', 'FLAG_OWN_CAR',
        'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
        'AMT_ANNUITY', 'AMT_GOODS_PRICE'],
       dtype='object'))

## Checking the structure: how train and test are told apart

The final matrix holds two kinds of row:
- training rows, where `TARGET` is present
- Kaggle test rows, where `TARGET` is null

The check below confirms the split is what it claims to be.

In [5]:
assert "TARGET" in df2.columns

n_train = int(df2["TARGET"].notna().sum())
n_test = int(df2["TARGET"].isna().sum())
n_train, n_test


(307507, 48744)

## Conclusion

One feature matrix (`features.parquet`), built reproducibly: aggregations per table, joins on `SK_ID_CURR`, train and test separated by `TARGET`.

It is the **single** source for everything downstream — the exploratory analysis in notebook 02, the training and MLflow tracking in notebook 03, and the serving pipeline in the second part of the project. One matrix rather than several is what keeps those three from drifting apart.